# Practical 8 (B3) - Data Cleaning and Preparation
### Analyzing Customer Churn in a Telecommunications Company

**Setup:** Import required libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set(style="whitegrid")
print("Libraries ready")

**Task 1:** Import the "Telecom_Customer_Churn.csv" dataset.

In [ ]:
url = "Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.head()

**Task 2:** Explore the dataset to understand its structure and content.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:\n", list(df.columns))
print("\nInfo:")
df.info()

In [ ]:
df.describe()

**Task 3:** Handle missing values in the dataset, deciding on an appropriate strategy.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values per column:")
print(df.isnull().sum())

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("\nMissing values after filling:")
print(df.isnull().sum().sum())

**Task 4:** Remove any duplicate records from the dataset.

In [ ]:
print("Rows before:", df.shape[0])
df = df.drop_duplicates()
print("Rows after removing duplicates:", df.shape[0])

**Task 5:** Check for inconsistent data, such as inconsistent formatting or spelling variations, and standardize it.

In [ ]:
df = df.replace("No internet service", "No")
df = df.replace("No phone service", "No")

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("Unique values in MultipleLines:", df["MultipleLines"].unique())
print("Unique values in OnlineSecurity:", df["OnlineSecurity"].unique())

**Task 6:** Convert columns to the correct data types as needed.

In [ ]:
df["SeniorCitizen"] = df["SeniorCitizen"].astype("category")
df["Churn"] = df["Churn"].astype("category")

print(df.dtypes)

**Task 7:** Identify and handle outliers in the data.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: {outliers} outliers detected")
    df[col] = df[col].clip(lower, upper)

plt.figure(figsize=(8, 4))
df[numeric_cols].boxplot()
plt.title("Numeric Columns After Handling Outliers")
plt.show()

**Task 8:** Perform feature engineering, creating new features that may be relevant to predicting customer churn.

In [ ]:
df["tenure_group"] = pd.cut(df["tenure"],
                            bins=[0, 12, 24, 48, 60, 72],
                            labels=["0-1yr", "1-2yr", "2-4yr", "4-5yr", "5-6yr"])

df["avg_monthly_spend"] = df["TotalCharges"] / (df["tenure"] + 1)

df[["tenure", "tenure_group", "TotalCharges", "avg_monthly_spend"]].head()

**Task 9:** Normalize or scale the data if necessary.

In [ ]:
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df_scaled[numeric_cols].head()

**Task 10:** Split the dataset into training and testing sets for further analysis.

In [ ]:
X = df_scaled.drop("Churn", axis=1)
y = df_scaled["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

**Task 11:** Export the cleaned dataset for future analysis or modeling.

In [ ]:
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
print("Saved as Telecom_Customer_Churn_Cleaned.csv in the notebook folder")